# Construct a 128-Student Fixed-Tolerance Course Dataset

This notebook creates a synthetic course-allocation preference dataset with:

- 128 students;
- heterogeneous preference dispersion, goals, constraints, and language styles;
- one **common** practical-indifference tolerance for every student;
- bundle utilities and the tolerance measured on one global scale;
- matched-size bundle comparisons, with balanced `A`, `B`, and `INDIFFERENT` labels.

No student-specific normalization is used. This is essential: normalizing every
student separately would erase the dispersion heterogeneity that the experiment
is designed to study.

In [ ]:
import json
import math
import random
import shutil
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 20260909
N_STUDENTS = 128
N_COURSES = 12
MAX_BUNDLE_SIZE = 5
COMMON_TOLERANCE = 0.12
QUERIES_PER_LABEL_PER_STUDENT = 8

random.seed(SEED)
rng = np.random.default_rng(SEED)

OUTPUT_DIR = Path("fixed_tolerance_course_data")
if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True)

## 1. Public course catalog

In [ ]:
COURSES = [
    (0, "Optimization for Decisions", "optimization", "A", "high", "mathematical optimization and resource allocation"),
    (1, "Data Analytics with Python", "analytics", "B", "medium", "predictive modeling, coding, and data analysis"),
    (2, "Behavioral Decision Making", "behavior", "C", "medium", "judgment, experiments, and behavioral science"),
    (3, "Entrepreneurial Strategy", "strategy", "A", "medium", "new ventures, competition, and market entry"),
    (4, "Financial Modeling", "finance", "B", "high", "valuation, spreadsheets, and corporate finance"),
    (5, "Machine Learning Applications", "analytics", "C", "high", "machine learning systems and applied projects"),
    (6, "Supply Chain Analytics", "operations", "A", "high", "inventory, logistics, and supply-chain decisions"),
    (7, "Negotiation", "behavior", "B", "medium", "negotiation practice and interpersonal decision making"),
    (8, "Platform and Marketplace Design", "strategy", "C", "high", "matching markets, platforms, and mechanism design"),
    (9, "Sustainable Operations", "operations", "A", "medium", "sustainability, circular systems, and operations"),
    (10, "Healthcare Operations", "operations", "B", "medium", "healthcare delivery, queues, and capacity planning"),
    (11, "Independent Research Seminar", "research", "C", "high", "faculty-guided research and academic writing"),
]

catalog = pd.DataFrame(
    COURSES,
    columns=["course_id", "course_name", "field", "time_slot", "workload", "description"],
)
assert len(catalog) == N_COURSES
catalog.to_csv(OUTPUT_DIR / "course_catalog.csv", index=False)
display(catalog)

## 2. Student heterogeneity and latent utilities

In [ ]:
GOAL_WEIGHTS = {
    "PhD preparation": {"research": 2.0, "optimization": 1.5, "analytics": 1.0, "operations": 0.8},
    "analytics career": {"analytics": 1.8, "optimization": 1.1, "finance": 0.6, "operations": 0.7},
    "consulting career": {"strategy": 1.5, "behavior": 1.1, "finance": 0.8, "operations": 0.5},
    "operations career": {"operations": 1.8, "optimization": 1.2, "analytics": 0.8},
    "entrepreneurship": {"strategy": 1.8, "behavior": 0.9, "finance": 0.6, "analytics": 0.4},
    "degree completion": {"behavior": 0.35, "strategy": 0.35, "operations": 0.25, "analytics": 0.15},
}

DISPERSION_TYPES = {
    "focused": 1.35,
    "moderate": 0.90,
    "flexible": 0.50,
}

LANGUAGE_STYLES = ["concise", "reflective", "career-focused", "constraint-focused"]
WORKLOAD_LEVEL = {"medium": 0.0, "high": 1.0}


def generate_student(student_id):
    goal = rng.choice(list(GOAL_WEIGHTS), p=[0.16, 0.20, 0.18, 0.18, 0.12, 0.16])
    dispersion_type = rng.choice(list(DISPERSION_TYPES), p=[0.34, 0.42, 0.24])
    sigma = DISPERSION_TYPES[dispersion_type]
    style = rng.choice(LANGUAGE_STYLES)
    preferred_slot = rng.choice(["A", "B", "C", "NONE"], p=[0.22, 0.22, 0.22, 0.34])
    workload_sensitivity = float(rng.uniform(0.05, 0.75))

    raw_scores = []
    for row in catalog.itertuples():
        goal_fit = GOAL_WEIGHTS[goal].get(row.field, 0.0)
        slot_fit = 0.35 if preferred_slot == row.time_slot else 0.0
        workload_cost = workload_sensitivity * WORKLOAD_LEVEL[row.workload]
        idiosyncratic = float(rng.normal(0.0, 0.42))
        raw_scores.append(goal_fit + slot_fit - workload_cost + idiosyncratic)

    # Center on the shared scale, then apply the student's dispersion.
    raw_scores = np.asarray(raw_scores)
    centered = raw_scores - raw_scores.mean()
    course_values = np.clip(sigma * centered, -2.4, 2.4)

    ranked = np.argsort(course_values)
    top = list(map(int, ranked[-4:]))
    bottom = list(map(int, ranked[:3]))

    # Relationships are sparse and use the same global utility scale.
    complements = []
    substitutes = []
    if rng.random() < 0.78:
        pair = tuple(sorted(rng.choice(top, size=2, replace=False).tolist()))
        complements.append({"pair": list(pair), "value": float(rng.uniform(0.18, 0.38) * sigma)})
    if rng.random() < 0.68:
        same_field_pairs = [
            (a, b) for a, b in combinations(range(N_COURSES), 2)
            if catalog.loc[a, "field"] == catalog.loc[b, "field"]
        ]
        if same_field_pairs:
            pair = same_field_pairs[int(rng.integers(len(same_field_pairs)))]
            substitutes.append({"pair": list(pair), "value": -float(rng.uniform(0.14, 0.32) * sigma)})

    return {
        "student_id": int(student_id),
        "goal": str(goal),
        "dispersion_type": str(dispersion_type),
        "dispersion_parameter": float(sigma),
        "language_style": str(style),
        "preferred_time_slot": str(preferred_slot),
        "workload_sensitivity": workload_sensitivity,
        "course_values": {str(j): float(course_values[j]) for j in range(N_COURSES)},
        "complements": complements,
        "substitutes": substitutes,
        "top_courses": top,
        "bottom_courses": bottom,
    }


students = [generate_student(i) for i in range(N_STUDENTS)]

## 3. Natural-language reports

In [ ]:
OPENERS = {
    "concise": [
        "I am trying to make a practical course plan.",
        "My priorities for next term are fairly straightforward.",
    ],
    "reflective": [
        "I have been thinking about how this term fits into my longer-run plans.",
        "I am still weighing several directions, but a few preferences stand out.",
    ],
    "career-focused": [
        "I want this schedule to support my next career step.",
        "I am choosing courses mainly with my post-graduation plans in mind.",
    ],
    "constraint-focused": [
        "I need a useful schedule that also remains manageable.",
        "Scheduling and workload matter alongside course content for me.",
    ],
}

POSITIVE = [
    "is especially attractive to me",
    "fits what I hope to learn",
    "would be a meaningful part of my schedule",
    "looks genuinely useful for my plans",
]
NEGATIVE = [
    "is not a priority for me",
    "does not fit my current direction very well",
    "would be difficult for me to justify taking",
    "is less relevant to what I need this term",
]


def cname(course_id):
    return catalog.loc[catalog.course_id == course_id, "course_name"].iloc[0]


def make_language_report(student):
    values = np.array([student["course_values"][str(j)] for j in range(N_COURSES)])
    order = np.argsort(values)
    strongest = list(map(int, order[-3:][::-1]))
    weakest = list(map(int, order[:2]))
    style = student["language_style"]

    sentences = [str(rng.choice(OPENERS[style]))]
    sentences.append(f"My main goal is {student['goal']}.")

    if student["dispersion_type"] == "focused":
        sentences.append(
            f"{cname(strongest[0])} {rng.choice(POSITIVE)}, and I would place it clearly ahead of {cname(weakest[0])}."
        )
        sentences.append(f"I am also quite positive about {cname(strongest[1])}.")
    elif student["dispersion_type"] == "moderate":
        sentences.append(
            f"I lean toward {cname(strongest[0])} and {cname(strongest[1])}, while {cname(weakest[0])} is less appealing."
        )
        sentences.append("Several of the remaining options are reasonably close for me.")
    else:
        sentences.append(
            f"{cname(strongest[0])} may be somewhat more useful than most alternatives, but I do not have many sharp course-by-course distinctions."
        )
        sentences.append("As long as the schedule works, several combinations would feel similarly satisfactory.")

    if values[weakest[0]] < -0.55:
        sentences.append(f"{cname(weakest[0])} {rng.choice(NEGATIVE)}.")

    if student["complements"]:
        a, b = student["complements"][0]["pair"]
        sentences.append(
            f"Taking {cname(a)} together with {cname(b)} would be particularly valuable because the two subjects reinforce each other."
        )

    if student["substitutes"]:
        a, b = student["substitutes"][0]["pair"]
        sentences.append(
            f"I would probably not take both {cname(a)} and {cname(b)} because they overlap for my purposes."
        )

    if student["preferred_time_slot"] != "NONE":
        sentences.append(f"When possible, I prefer courses in time slot {student['preferred_time_slot']}.")
    if student["workload_sensitivity"] > 0.55:
        sentences.append("I also want to avoid a schedule made up entirely of high-workload courses.")
    elif student["workload_sensitivity"] < 0.22:
        sentences.append("I am comfortable accepting a demanding workload for sufficiently relevant courses.")

    return " ".join(sentences)


reports = []
for student in students:
    report = make_language_report(student)
    student["language_report"] = report
    reports.append({"student_id": student["student_id"], "language_report": report})

reports_df = pd.DataFrame(reports)
reports_df.to_csv(OUTPUT_DIR / "student_reports.csv", index=False)

for row in reports_df.sample(5, random_state=SEED).itertuples():
    print(f"Student {row.student_id}:\n{row.language_report}\n")

## 4. Bundle utility and balanced comparison benchmark

In [ ]:
def bundle_utility(student, bundle):
    # Return utility on one common scale; never normalize by student.
    bundle = tuple(sorted(map(int, bundle)))
    value = sum(student["course_values"][str(j)] for j in bundle)
    pair_set = set(combinations(bundle, 2))

    for relation in student["complements"]:
        if tuple(relation["pair"]) in pair_set:
            value += relation["value"]
    for relation in student["substitutes"]:
        if tuple(relation["pair"]) in pair_set:
            value += relation["value"]

    # MAX_BUNDLE_SIZE is global, so dispersion remains comparable across students.
    return float(value / MAX_BUNDLE_SIZE)


def sample_bundle_pair():
    size = int(rng.integers(2, MAX_BUNDLE_SIZE + 1))
    a = tuple(sorted(rng.choice(N_COURSES, size=size, replace=False).tolist()))
    b = tuple(sorted(rng.choice(N_COURSES, size=size, replace=False).tolist()))
    return a, b


def true_label_from_difference(difference):
    if abs(difference) <= COMMON_TOLERANCE:
        return "INDIFFERENT"
    return "A" if difference > 0 else "B"


query_rows = []
query_id = 0
target_labels = ("A", "B", "INDIFFERENT")

for student in students:
    pools = {label: [] for label in target_labels}
    attempts = 0
    while min(len(pools[label]) for label in target_labels) < QUERIES_PER_LABEL_PER_STUDENT:
        attempts += 1
        if attempts > 100000:
            raise RuntimeError(f"Could not balance queries for student {student['student_id']}.")
        bundle_a, bundle_b = sample_bundle_pair()
        if bundle_a == bundle_b:
            continue
        utility_a = bundle_utility(student, bundle_a)
        utility_b = bundle_utility(student, bundle_b)
        difference = utility_a - utility_b
        label = true_label_from_difference(difference)
        if len(pools[label]) < QUERIES_PER_LABEL_PER_STUDENT:
            pools[label].append((bundle_a, bundle_b, utility_a, utility_b, difference))

    for label in target_labels:
        for bundle_a, bundle_b, utility_a, utility_b, difference in pools[label]:
            query_rows.append({
                "query_id": query_id,
                "student_id": student["student_id"],
                "bundle_a": json.dumps(list(bundle_a)),
                "bundle_b": json.dumps(list(bundle_b)),
                "bundle_size": len(bundle_a),
                "true_utility_a": utility_a,
                "true_utility_b": utility_b,
                "true_difference": difference,
                "true_gap": abs(difference),
                "true_direction": "A" if difference > 0 else "B",
                "true_label": label,
                "common_tolerance": COMMON_TOLERANCE,
            })
            query_id += 1

queries_df = pd.DataFrame(query_rows)
queries_df.to_csv(OUTPUT_DIR / "bundle_queries_with_truth.csv", index=False)

print("Queries:", len(queries_df))
display(pd.crosstab(queries_df.bundle_size, queries_df.true_label))

## 5. Save public reports, private ground truth, and metadata

In [ ]:
private_students = []
for student in students:
    private_students.append({
        key: value for key, value in student.items()
        if key != "language_report"
    })

(OUTPUT_DIR / "student_ground_truth.json").write_text(
    json.dumps({"students": private_students}, indent=2),
    encoding="utf-8",
)

metadata = {
    "seed": SEED,
    "n_students": N_STUDENTS,
    "n_courses": N_COURSES,
    "maximum_bundle_size": MAX_BUNDLE_SIZE,
    "common_tolerance": COMMON_TOLERANCE,
    "queries_per_label_per_student": QUERIES_PER_LABEL_PER_STUDENT,
    "query_labels": ["A", "B", "INDIFFERENT"],
    "utility_definition": "(sum of course values plus pair interactions) / global MAX_BUNDLE_SIZE",
    "normalization_note": "No student-specific normalization is used.",
    "design_note": "Students have heterogeneous utility dispersion but share one practical-indifference tolerance.",
}
(OUTPUT_DIR / "generation_metadata.json").write_text(
    json.dumps(metadata, indent=2), encoding="utf-8"
)

summary = pd.DataFrame({
    "student_id": [s["student_id"] for s in students],
    "goal": [s["goal"] for s in students],
    "dispersion_type": [s["dispersion_type"] for s in students],
    "dispersion_parameter": [s["dispersion_parameter"] for s in students],
    "course_value_std": [np.std(list(s["course_values"].values())) for s in students],
})
summary.to_csv(OUTPUT_DIR / "student_summary.csv", index=False)
display(summary.groupby("dispersion_type")["course_value_std"].describe())

## 6. Validation plots and export

In [ ]:
assert reports_df.student_id.nunique() == N_STUDENTS
assert queries_df.student_id.nunique() == N_STUDENTS
assert queries_df.groupby(["student_id", "true_label"]).size().eq(QUERIES_PER_LABEL_PER_STUDENT).all()
assert np.allclose(queries_df.common_tolerance, COMMON_TOLERANCE)
assert (queries_df.bundle_a.apply(len) > 0).all()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for name, group in summary.groupby("dispersion_type"):
    axes[0].hist(group.course_value_std, bins=12, alpha=0.55, label=name)
axes[0].set_title("Realized preference dispersion")
axes[0].set_xlabel("Standard deviation of course values")
axes[0].legend()

queries_df.true_gap.hist(bins=30, ax=axes[1])
axes[1].axvline(COMMON_TOLERANCE, color="red", linestyle="--", label="Common tolerance")
axes[1].set_title("True bundle-value gaps")
axes[1].set_xlabel("Absolute utility gap")
axes[1].legend()
plt.tight_layout()
plt.show()

archive_path = shutil.make_archive(str(OUTPUT_DIR), "zip", OUTPUT_DIR)
print("Saved:", archive_path)

try:
    from google.colab import files
    files.download(archive_path)
except ImportError:
    print("Not running in Colab; the ZIP remains in the current directory.")